In [ ]:
#| default_exp core

# SolveitCDP API
> Source and API details

In [ ]:
#| export
from fastcore.utils import *
import json, asyncio, inspect, base64
from contextlib import asynccontextmanager
from dialoghelper import *

In [ ]:
from fastcore.test import test_eq

In [ ]:
#| export
if '__file__' not in globals():
    _root = Path.cwd()
    if not (Path.cwd()/'pyproject.toml').exists(): _root = _root.parent
    __file__ = _root/'solvecdp'/'core.py'

_path = Path(__file__).parent
_bp = (_path/'browser_protocol.json').read_json()
_jp = (_path/'js_protocol.json').read_json()
_cdp_domains = _bp['domains'] + _jp['domains']

In [ ]:
len(_cdp_domains), [d['domain'] for d in _cdp_domains[:5]]

(55, ['Accessibility', 'Animation', 'Audits', 'Autofill', 'BackgroundService'])

In [ ]:
#| export
def cdp_search(q:str):
    "Search CDP domains and commands by name or description"
    q = q.lower()
    res = []
    for d in _cdp_domains:
        for cmd in d.get('commands', []):
            desc = cmd.get('description', '')
            if q in cmd['name'].lower() or q in desc.lower():
                res.append(f"{d['domain']}.{cmd['name']}: {desc[:120]}")
        for evt in d.get('events', []):
            desc = evt.get('description', '')
            if q in evt['name'].lower() or q in desc.lower():
                res.append(f"  evt {d['domain']}.{evt['name']}: {desc[:120]}")
    return '\n'.join(res)

In [ ]:
cdp_search('target')[:100]

'Audits.checkFormsIssues: Runs the form issues check for the target page. Found issues are reported\nu'

solvecdp relies on the solveit-chrome extension being available - let's check it:

In [ ]:
test_eq((await event_get_a('ext-ping')).result, 'pong')

Let's try using the extension cdp directly:

In [ ]:
r = await event_get_a('cdp-new-tab', url='https://example.com')
tid = r.result['tabId']
await asyncio.sleep(0.5)
tid

895201191

We can add a little convenience for it:

In [ ]:
#| export
async def js_cdp(method, tabId=None, **params):
    res = (await event_get_a('cdp-send', tabId=tabId, method=method, params=params)).result
    return first(res.values()) if res and len(res)==1 else res

In [ ]:
await js_cdp('Runtime.evaluate', tabId=tid, expression='document.title')

```python
{'type': 'string', 'value': '6. Example Domain'}
```

In [ ]:
#| export
def _lower1(s): return s[0].lower() + s[1:]
def _upper1(s): return s[0].upper() + s[1:]

In [ ]:
#| export
class JsCDP:
    "Chrome DevTools Protocol via JS bridge with event support"
    def __init__(self, tid): self.tid,self._sub_id = tid,0

    @classmethod
    async def new(cls, tid=None, url='about:blank', active=False):
        if tid: await event_get_a('cdp-attach', tabId=tid)
        else:
            r = await event_get_a('cdp-new-tab', url=url, active=active)
            tid = r.result['tabId']
        self = cls(tid)
        await self('Page.enable')
        await self('Network.enable')
        await self('Page.setLifecycleEventsEnabled', enabled=True)
        return self

    async def __call__(self, method, **params): return await js_cdp(method, tabId=self.tid, **params)

    def __getattr__(self, domain):
        if domain.startswith('_'): raise AttributeError()
        return JsCDPDomain(self, _upper1(domain))

    async def close(self): await event_get_a('cdp-close-tab', tabId=self.tid)
    async def detach(self): await event_get_a('cdp-detach', tabId=self.tid)
    async def __aenter__(self): return self
    async def __aexit__(self, *exc): await self.detach()

    def __del__(self):
        try: asyncio.get_event_loop().create_task(self.detach())
        except Exception: pass

In [ ]:
jc = await JsCDP.new()

We can subscribe to events thanks to the `solveit-chrome` extension bridge:

In [ ]:
#| export
class JsSub:
    def __init__(self, cdp, events): self.cdp,self.events,self.seq = cdp,events,0
    async def __call__(self, timeout=10):
        self.seq += 1
        return await pop_data_a(f'{self.sub_id}:{self.seq}', timeout)
    async def __aenter__(self):
        self.cdp._sub_id += 1
        self.sub_id = f'jsc_{self.cdp.tid}_{self.cdp._sub_id}'
        await event_get_a('cdp-subscribe', tabId=self.cdp.tid, subId=self.sub_id, events=self.events)
        return self
    async def __aexit__(self, *exc): await self.cdp.unsubscribe(self.sub_id)

@patch
async def unsubscribe(self:JsCDP, sub_id): await event_get_a('cdp-unsubscribe', tabId=self.tid, subId=sub_id)

@patch
def on(self:JsCDP, *events): return JsSub(self, events)

In [ ]:
async with jc.on('Page.loadEventFired') as sub:
    await jc('Page.navigate', url='https://example.com')
    evt = await sub()

In [ ]:
await jc.close()

In [ ]:
#| export
_domains = {d['domain']: d for d in _cdp_domains}

def _find_cmd(domain, method):
    d = _domains.get(domain, {})
    return first(c for c in d.get('commands', []) if c['name'] == method)

def _cmd_sig(cmd):
    _P = inspect.Parameter
    ps = [_P(p['name'], _P.KEYWORD_ONLY, default=None if p.get('optional') else _P.empty) for p in cmd.get('parameters', [])]
    return inspect.Signature(ps)

def _cmd_doc(domain, method, cmd):
    doc = f"{domain}.{method}"
    if cmd.get('description'): doc += f" - {cmd['description']}"
    for p in cmd.get('parameters', []):
        opt = ' (optional)' if p.get('optional') else ''
        doc += f"\n  {p['name']}{opt}: {p.get('description', '')}"
    return doc

In [ ]:
#| export
class JsCDPDomain:
    def __init__(self, cdp, domain): store_attr()
    def __getattr__(self, method):
        if method.startswith('_'): raise AttributeError()
        return JsCDPMethod(self.cdp, self.domain, method)
    def __dir__(self):
        d = _domains.get(self.domain, {})
        return [c['name'] for c in d.get('commands', [])] + [e['name'] for e in d.get('events', [])]

class JsCDPMethod:
    def __init__(self, cdp, domain, method):
        store_attr()
        if (cmd := _find_cmd(domain, method)):
            self.__doc__ = _cmd_doc(domain, method, cmd)
            self.__signature__ = _cmd_sig(cmd)
    async def __call__(self, **kw): return await self.cdp(f'{self.domain}.{self.method}', **kw)

In [ ]:
#| export
@patch
async def wait_for(self:JsCDP, expr, timeout=10):
    "Wait for JS expression to be truthy, return its value"
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        r = await self.eval(expr)
        if r: return r
        await asyncio.sleep(0.05)
    raise TimeoutError(f'Timed out waiting for: {expr}')

@patch
async def wait_for_selector(self:JsCDP, sel, timeout=10):
    "Wait for CSS selector to match an element"
    return await self.wait_for(f'!!document.querySelector("{sel}")', timeout)

@patch
async def eval(self:JsCDP, expr):
    res = await self.runtime.evaluate(expression=expr)
    return res['value'] if res else res

In [ ]:
jc = await JsCDP.new()

In [ ]:
await jc.page.navigate(url='https://example.com')
await jc.wait_for('document.title')

'Example Domain'

In [ ]:
await jc.close()

In [ ]:
#| export
@patch
async def wait_for_ready(self:JsCDP, timeout=10, idle_ms=500):
    "Wait until network is idle for idle_ms"
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        async with self.on('Network.requestWillBeSent') as q:
            r = await q(idle_ms/1000)
            if not r or r.get('js_status')=='timeout': return
    raise TimeoutError('Timed out waiting for network idle')

@patch
async def goto(self:JsCDP, url, **kwargs):
    "Navigate to url and wait for load+idle"
    await self.page.navigate(url=url)
    await self.wait_for_ready(**kwargs)

In [ ]:
jc = await JsCDP.new()

In [ ]:
await jc.goto('https://example.com')
await jc.eval('document.title')

'6. Example Domain'

In [ ]:
#| export
@patch(as_prop=True)
async def pages(self:JsCDP):
    targets = (await event_get_a('cdp-get-targets')).result
    return [t for t in targets if t.get('type') == 'page']

In [ ]:
pages = await jc.pages
for p in pages[:3]: print(f"{p.get('tabId', '?'):>10} {p.get('title', '')[:60]}")
pg = first(p for p in pages if 'Example' in p['title'])
pg['url']

 895199784 8. Database Transactions — PlanetScale
 895200613 2. l:SolveIT - nbs◀️dialoghelper◀️aai-ws
 895199539 2. [AnswerDotAI/safepyrun] Run failed: Deploy to GitHub Page


'https://example.com/'

In [ ]:
jc2 = await JsCDP.new(pg['tabId'])
await jc2.eval('document.title')

'5. Example Domain'

In [ ]:
#| export
@patch
async def screenshot(self:JsCDP):
    from IPython.display import Image
    b64 = await self.page.captureScreenshot(format='png')
    return Image(base64.b64decode(b64))

In [ ]:
# await jc2.screenshot()

In [ ]:
await jc2.close()

## LLMs and accessibility

In [ ]:
page = await JsCDP.new(url='https://httpbin.org/forms/post')
await page.eval('document.title')

'5. httpbin.org/forms/post'

In [ ]:
await page.accessibility.enable()
tree = await page.accessibility.getFullAXTree()
len(tree)

90

In [ ]:
tree[0]

```python
{ 'backendDOMNodeId': 14,
  'childIds': ['15'],
  'chromeRole': {'type': 'internalRole', 'value': 144},
  'frameId': '92354B6BB2A591B3DDB0267C24E21365',
  'ignored': False,
  'name': { 'sources': [{'attribute': 'aria-labelledby', 'type': 'relatedElement'}, {'attribute': 'aria-label', 'type': 'attribute'}, {'attribute': 'aria-label', 'superseded': True, 'type': 'attribute'}, {'nativeSource': 'title', 'type': 'relatedElement', 'value': {'type': 'computedString', 'value': '5. httpbin.org/forms/post'}}],
            'type': 'computedString',
            'value': '5. httpbin.org/forms/post'},
  'nodeId': '14',
  'properties': [{'name': 'focusable', 'value': {'type': 'booleanOrUndefined', 'value': True}}, {'name': 'url', 'value': {'type': 'string', 'value': 'https://httpbin.org/forms/post'}}],
  'role': {'type': 'internalRole', 'value': 'RootWebArea'}}
```

In [ ]:
await page.close()

In [ ]:
#| export
class AXNode:
    "Chrome accessibility tree node with compact repr"
    def __init__(self, raw):
        self.role = nested_idx(raw, 'role', 'value') or ''
        self.name = nested_idx(raw, 'name', 'value') or ''
        self.props = {p['name']: nested_idx(p, 'value', 'value') for p in raw.get('properties', [])}
        self.children = []
        self.node_id = raw.get('nodeId')
        self.backend_id = raw.get('backendDOMNodeId')
        self.ignored = raw.get('ignored', False)
        self._raw = raw

    def _repr_markdown_(self, depth=0):
        pre = '  ' * depth
        ps = ''.join(f' `{k}={v}`' for k,v in self.props.items() if v not in (False, None, '', 'false'))
        bid = f' [#{self.backend_id}]' if self.backend_id else ''
        line = f'{pre}- **{self.role}** "{self.name}"{ps}{bid}'
        return '\n'.join([line] + [c._repr_markdown_(depth+1) for c in self.children])

    __str__ = __repr__ = _repr_markdown_

class AXTree(AXNode):
    def _repr_markdown_(self): return f'<div class="prose">\n\n{super()._repr_markdown_(0)}\n\n</div>'

In [ ]:
#| export
def _simplify(node):
    node.children = [n for c in node.children for n in _simplify(c)]
    if node.role in ('none', 'generic', 'paragraph') and not node.name:
        if not node.children: return []
        return node.children  # splice children up
    return [node]

def build_ax_tree(nodes):
    "Build AXNode tree from flat CDP accessibility node list"
    by_id = {}
    for raw in nodes:
        n = AXNode(raw)
        by_id[n.node_id] = n
    for raw in nodes:
        parent = by_id.get(raw.get('nodeId'))
        for cid in raw.get('childIds', []):
            if (child := by_id.get(cid)): parent.children.append(child)
    if not nodes: return
    root = by_id.get(nodes[0]['nodeId'])
    if not (result := _simplify(root)): return None
    result = result[0]
    result.__class__ = AXTree
    return result

In [ ]:
#| export
@patch
async def ax_tree(self:JsCDP, sid=None):
    "Get accessibility tree for session"
    await self.accessibility.enable(sid=sid)
    return build_ax_tree(await self.accessibility.getFullAXTree(sid=sid))

In [ ]:
page = await JsCDP.new(url='https://httpbin.org/forms/post')

In [ ]:
root = await page.ax_tree()
root

<div class="prose">

- **RootWebArea** "5. httpbin.org/forms/post" `focusable=True` `url=https://httpbin.org/forms/post` [#14]
  - **LabelText** "" [#20]
    - **StaticText** "Customer name: " [#62]
      - **InlineTextBox** "Customer name: "
    - **textbox** "Customer name: " `focusable=True` `editable=plaintext` `settable=True` [#2]
  - **LabelText** "" [#23]
    - **StaticText** "Telephone: " [#63]
      - **InlineTextBox** "Telephone: "
    - **textbox** "Telephone: " `focusable=True` `editable=plaintext` `settable=True` [#3]
  - **LabelText** "" [#26]
    - **StaticText** "E-mail address: " [#64]
      - **InlineTextBox** "E-mail address: "
    - **textbox** "E-mail address: " `focusable=True` `editable=plaintext` `settable=True` [#4]
  - **group** "Pizza Size" [#28]
    - **Legend** "" [#29]
      - **StaticText** "Pizza Size" [#65]
        - **InlineTextBox** "Pizza Size"
    - **radio** " Small" `focusable=True` [#6]
    - **radio** " Medium" `focusable=True` [#7]
    - **radio** " Large" `focusable=True` [#8]
  - **group** "Pizza Toppings" [#36]
    - **Legend** "" [#37]
      - **StaticText** "Pizza Toppings" [#69]
        - **InlineTextBox** "Pizza Toppings"
    - **checkbox** " Bacon" `focusable=True` [#9]
    - **checkbox** " Extra Cheese" `focusable=True` [#10]
    - **checkbox** " Onion" `focusable=True` [#11]
    - **checkbox** " Mushroom" `focusable=True` [#12]
  - **LabelText** "" [#47]
    - **StaticText** "Preferred delivery time: " [#74]
      - **InlineTextBox** "Preferred delivery time: "
    - **InputTime** "Preferred delivery time: " `focusable=True` `settable=True` [#13]
      - **spinbutton** "Hours Hours" `focusable=True` `settable=True` `valuemin=1` `valuemax=12` [#51]
        - **StaticText** "--" [#75]
          - **InlineTextBox** "--"
      - **StaticText** ":" [#76]
        - **InlineTextBox** ":"
      - **spinbutton** "Minutes Minutes" `focusable=True` `settable=True` `valuemax=59` [#53]
        - **StaticText** "--" [#77]
          - **InlineTextBox** "--"
      - **StaticText** " " [#78]
        - **InlineTextBox** " "
      - **spinbutton** "AM/PM AM/PM" `focusable=True` `settable=True` `valuemin=1` `valuemax=2` [#55]
        - **StaticText** "--" [#79]
          - **InlineTextBox** "--"
      - **button** "Show time picker" `focusable=True` `hasPopup=menu` [#56]
  - **LabelText** "" [#58]
    - **StaticText** "Delivery instructions: " [#80]
      - **InlineTextBox** "Delivery instructions: "
    - **textbox** "Delivery instructions: " `focusable=True` `editable=plaintext` `settable=True` `multiline=True` [#5]
  - **button** "Submit order" `focusable=True` [#61]
    - **StaticText** "Submit order" [#81]
      - **InlineTextBox** "Submit order"

</div>

In [ ]:
#| export
@patch
def find(self:AXNode, role=None, name=None):
    "Find first descendant matching role and/or name substring"
    if (not role or role == self.role) and (not name or name in self.name): return self
    for c in self.children:
        if (r := c.find(role, name)): return r

@patch
def find_id(self:AXNode, role=None, name=None):
    "Find first descendant matching role and/or name substring"
    res = self.find(role, name)
    return res.backend_id if res else None

@patch
def find_all(self:AXNode, role=None, name=None):
    "Find all descendants matching role and/or name substring"
    res = []
    if (not role or role == self.role) and (not name or name in self.name): res.append(self)
    for c in self.children: res += c.find_all(role, name)
    return res

In [ ]:
nmid = root.find_id('textbox', 'Customer name')
phid = root.find_id('textbox', 'Telephone')
nmid

2

In [ ]:
#| export
@patch
async def js_node(self:JsCDP, fn, backendNodeId, sid=None):
    obj = await self.DOM.resolveNode(sid=sid, backendNodeId=backendNodeId)
    return await self.runtime.callFunctionOn(sid=sid, functionDeclaration=fn,
                                              objectId=obj['objectId'])

@patch
async def js_node_run(self:JsCDP, code, backendNodeId, sid=None):
    return await self.js_node(f'function() {{ {code} }}', backendNodeId, sid=sid)

@patch
async def click(self:JsCDP, backendNodeId, sid=None):
    await self.js_node_run('this.click()', backendNodeId, sid=sid)

In [ ]:
await page.DOM.focus(backendNodeId=nmid)
await page.input.insertText(text='Jeremy Howard')

await page.DOM.focus(backendNodeId=phid)
await page.input.insertText(text='555-1234')

```python
{}
```

In [ ]:
await page.click(root.find_id('radio', 'Large'))
await page.click(root.find_id('checkbox', 'Extra Cheese'))

In [ ]:
#| export
@patch
async def fill_text(self:JsCDP, backendNodeId, text, sid=None):
    await self.DOM.focus(sid=sid, backendNodeId=backendNodeId)
    await self.input.insertText(sid=sid, text=text)

In [ ]:
await page.fill_text(root.find_id('textbox', 'Delivery'), 'Ring the doorbell twice')

In [ ]:
await page.js_node_run('this.value = "18:30"', root.find_id('InputTime', 'delivery time'));

In [ ]:
#| export
@patch
async def click_and_wait(self:JsCDP, backendNodeId, **kwargs):
    "Click element and wait for load+idle"
    await self.js_node_run('this.click()', backendNodeId)
    await self.wait_for_ready(**kwargs)

In [ ]:
await page.click_and_wait(root.find_id('button', 'Submit order'))

In [ ]:
#| export
def cdp_yolo():
    "Allow all CDP classes in safepyrun"
    from safepyrun import allow
    allow({JsCDP:..., JsCDPDomain:..., JsCDPMethod:...})
    allow({AXNode:..., AXTree:...})